# 01 — M2: Küçük I-JEPA eğitimi ve DINO karşılaştırması

Burada iki ölçek ayrılır:

- **Hızlı mekanizma deneyi:** sentetik görüntü, küçük ViT, CPU'da dakikalar.
- **Kaggle deneyi:** `configs/ijepa_tiny.yaml`, Imagenette-160, 30 epoch ve ablation'lar.

Hızlı deney paper sonucu değildir. Asıl öğrenme hedefi loss'un tek başına neden yeterli olmadığını; `feature std`, cosine similarity ve effective rank ile collapse'ın nasıl izlendiğini görmek.

In [ ]:
from pathlib import Path
import os, sys
override = os.environ.get('JEPA_LAB_ROOT')
candidates = ([Path(override)] if override else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/jepa-study'), Path('/kaggle/working/I-JEPA'), Path('/content/jepa-study'), Path('/content/I-JEPA')]
ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Repo bulunamadı; JEPA_LAB_ROOT değişkenini ayarlayın.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repo:', ROOT)

In [ ]:
from pprint import pprint
import torch
from torch.utils.data import DataLoader, TensorDataset
from jepa_lab.device import seed_everything, select_device
from jepa_lab.runlog import load_yaml

seed_everything(42)
device = select_device('auto')
config = load_yaml(ROOT / 'configs/ijepa_tiny.yaml')
print('device:', device)
pprint(config)

## Tahmin 1 — Loss düşerse collapse bitmiş midir?

Tüm görüntülere aynı sabit vektörü veren encoder ile predictor düşük loss üretebilir mi? Böyle bir durumda patch feature standard deviation, örnekler arası cosine ve covariance effective rank nasıl davranır? Tahmininizi yazın.

In [ ]:
from jepa_lab.metrics import measure_collapse

healthy = torch.randn(16, 64, 32)
collapsed = torch.ones(16, 64, 32)
print('healthy  :', measure_collapse(healthy))
print('collapsed:', measure_collapse(collapsed))

### İncele 1

Collapse tek sayı değildir. Standard deviation sıfıra yaklaşırken cosine bire yaklaşır ve effective rank çöker. Küçük veri/bütçede değerleri paper tablosuyla değil, **aynı kod ve seed'deki random encoder** ile kıyaslayın.

## 10-step hızlı mekanizma eğitimi

### Tahmin 2

İlk ve son loss'un sırası hakkında tahmin yapın. Sadece 10 adım ve rastgele görüntüler olduğu için representation kalitesi hakkında hangi iddiayı **yapamayız**?

In [ ]:
from jepa_lab.image_jepa import ImageJEPA
from jepa_lab.masking import MultiBlockMasker
from jepa_lab.experiments import train_image_steps

generator = torch.Generator().manual_seed(42)
toy_images = torch.randn(128, 3, 64, 64, generator=generator)
toy_labels = torch.arange(128) % 10
toy_loader = DataLoader(TensorDataset(toy_images, toy_labels), batch_size=8, shuffle=False)
model = ImageJEPA(
    image_size=64, patch_size=8, embed_dim=64,
    encoder_depth=2, encoder_heads=4,
    predictor_dim=64, predictor_depth=2, predictor_heads=4,
)
masker = MultiBlockMasker((8, 8), num_targets=4, target_scale=(0.15, 0.20), context_scale=(0.85, 1.0))
optimizer = torch.optim.AdamW(model.trainable_parameters(), lr=1e-3)
quick_metrics = train_image_steps(
    model, toy_loader, masker, optimizer, steps=10, device=device,
    ema_start=0.996, ema_end=1.0, seed=42,
)
pprint(quick_metrics.as_dict())

# Smoke koşusunun gerçekten devam ettirilebilir olduğunu da doğrula.
from jepa_lab.checkpointing import load_training_checkpoint, save_training_checkpoint
smoke_checkpoint = save_training_checkpoint(
    ROOT / 'runs/ijepa_smoke.pt', model, optimizer, step=10,
    metadata={'seed': 42, 'scope': 'M2 ten-step smoke'},
)
resume = load_training_checkpoint(smoke_checkpoint, model, optimizer)
print({'resumed_step': resume.step, 'sha256': resume.sha256, 'metadata': resume.metadata})
assert resume.step == 10 and not resume.missing_keys and not resume.unexpected_keys

### İncele 2

Loss gürültülü olabilir ve son adım ilk adımdan yüksek çıkabilir; bu smoke test'in başarısız olduğu anlamına gelmez. Başarı burada sonlu tensor, doğru gradient sınırı ve çalışan EMA'dır. Representation iddiası için Imagenette, random baseline, k-NN ve frozen linear probe gerekir.

## Küçük ablation: dört target, tek target ve momentum-0 teacher

### Tahmin 3

Momentum `0` target'ı her adım online encoder'a eşitler. Bu, EMA teacher'ın yavaş hedef üretme rolünü nasıl değiştirir? 20-step küçük sonuçların paper sıralamasını neden birebir üretmesini beklememeliyiz?

In [ ]:
from jepa_lab.masking import RandomPatchMasker

def run_toy_ablation(name, *, targets=4, ema_start=0.996, mask_kind='multiblock', target_masking='output', seed=42):
    seed_everything(seed)
    candidate = ImageJEPA(
        image_size=64, patch_size=8, embed_dim=48, encoder_depth=1, encoder_heads=3,
        predictor_dim=48, predictor_depth=1, predictor_heads=3, target_masking=target_masking,
    )
    masker_type = MultiBlockMasker if mask_kind == 'multiblock' else RandomPatchMasker
    candidate_masker = masker_type(
        (8, 8), num_targets=targets, target_scale=(0.15, 0.20), context_scale=(0.85, 1.0)
    )
    candidate_opt = torch.optim.AdamW(candidate.trainable_parameters(), lr=1e-3)
    result = train_image_steps(
        candidate, toy_loader, candidate_masker, candidate_opt, steps=20, device=device,
        ema_start=ema_start, ema_end=ema_start, seed=seed,
    )
    return {'name': name, **result.as_dict()}

toy_ablation = [
    run_toy_ablation('four-target + EMA', targets=4, ema_start=0.996),
    run_toy_ablation('one-target + EMA', targets=1, ema_start=0.996),
    run_toy_ablation('random-patch + EMA', mask_kind='random'),
    run_toy_ablation('target input masking + EMA', target_masking='input'),
    run_toy_ablation('four-target + momentum-0', targets=4, ema_start=0.0),
]
pprint(toy_ablation)

## Kaggle uzun koşusu — Imagenette-160

Aşağıdaki hücre **opt-in**'dir: ağı indirir ve effective batch hesabıyla 30 epoch çalıştırır. Kaggle'da önce 10-step smoke, sonra 128 görüntü/300-step overfit, en son 30 epoch baseline çalıştırın. Effective batch `32×4=128`; OOM olursa microbatch'i yarıya indirip accumulation'ı ikiye katlayın.

Bu runner, pinlenmiş upstream'in gerçek `vit_tiny`, `vit_predictor`, `MaskCollator` ve `apply_masks` bileşenlerini ayrı process'te kullanır; paper ölçeğinde reproduction değildir.

In [ ]:
import subprocess
from jepa_lab.datasets import download_imagenette

RUN_IMAGENETTE_SMOKE = False
RUN_IMAGENETTE_OVERFIT = False
RUN_LONG_IMAGENETTE = False
RUN_IMAGENETTE_ABLATIONS = False

if RUN_IMAGENETTE_SMOKE or RUN_IMAGENETTE_OVERFIT or RUN_LONG_IMAGENETTE or RUN_IMAGENETTE_ABLATIONS:
    data_root = download_imagenette(ROOT / 'data')

def run_official_ijepa(name, *extra):
    command = [
        sys.executable, str(ROOT / 'scripts/official_ijepa_train.py'),
        '--device', str(device), '--data-root', str(data_root),
        '--image-size', '224', '--patch-size', '16', '--model', 'vit_tiny',
        '--predictor-dim', '192', '--predictor-depth', '4',
        '--batch-size', '32', '--gradient-accumulation', '4', '--seed', '42',
        '--checkpoint-output', str(ROOT / f'runs/{name}.pt'),
        '--summary-output', str(ROOT / f'runs/{name}.json'),
        *extra,
    ]
    subprocess.run(command, cwd=ROOT, check=True)

if RUN_IMAGENETTE_SMOKE:
    run_official_ijepa('ijepa_imagenette_smoke_10', '--steps', '10')

if RUN_IMAGENETTE_OVERFIT:
    run_official_ijepa('ijepa_overfit_128x300', '--max-images', '128', '--steps', '300')

if RUN_LONG_IMAGENETTE:
    run_official_ijepa('ijepa_baseline_30e', '--epochs', '30', '--evaluate-probes')

if RUN_IMAGENETTE_ABLATIONS:
    ablations = {
        'one_target': ('--epochs', '10', '--num-targets', '1'),
        'random_patch_mask': ('--epochs', '10', '--mask-kind', 'random-patch'),
        'target_input_masking': ('--epochs', '10', '--target-masking', 'input'),
        'zero_momentum_teacher': (
            '--epochs', '10', '--ema-start', '0', '--ema-end', '0'
        ),
    }
    for ablation_name, arguments in ablations.items():
        run_official_ijepa(f'ijepa_ablation_{ablation_name}', *arguments)

if not (RUN_IMAGENETTE_SMOKE or RUN_IMAGENETTE_OVERFIT or RUN_LONG_IMAGENETTE or RUN_IMAGENETTE_ABLATIONS):
    print('Atlandı. Kaggle GPU aşaması için istediğiniz RUN_* bayrağını açın.')

## k-NN ve frozen linear probe sonucu

Aynı train/validation split ve seed ile hem eğitilmiş encoder hem de aynı mimaride random encoder için çalıştırın. Kabul ölçütü: eğitilmiş modelin k-NN/linear sonucu random encoder'dan en az 5 yüzde puan yüksek.

In [ ]:
import json

baseline_summary = ROOT / 'runs/ijepa_baseline_30e.json'
if baseline_summary.is_file():
    baseline_result = json.loads(baseline_summary.read_text())
    pprint(baseline_result['representation_evaluation'])
else:
    print('30-epoch resmî runner --evaluate-probes ile trained/random sonucu aynı JSON dosyasına yazar.')

## DINO karşılaştırması — resmî frozen ViT-S/16

DINO burada JEPA değildir ve sıfırdan eğitilmez. İki resmî script ortak `shared-image-scene-v1` görüntüsünü ve aynı crop/renk/occlusion dönüşümlerini kullanır; yalnız her modelin kendi preprocessing'i uygulanır. Hücre DINO'yu indirir ve yerel I-JEPA checkpoint'ini ister; bayrak kapalıyken çalışmaz.

### Tahmin 4

DINO'nun view-invariance objective'i crop/renk dönüşümlerine neden güçlü olabilir? I-JEPA'nın predictor'ı ise target konumunu neden bilmek zorundadır?

In [ ]:
RUN_DINO_IJEPA_VIEWS = False
if RUN_DINO_IJEPA_VIEWS:
    ijepa_checkpoint = ROOT / 'checkpoints/IN1K-vit.h.14-300e.pth.tar'
    if not ijepa_checkpoint.is_file():
        raise FileNotFoundError('Önce resmî I-JEPA ViT-H/14 checkpointini indirin.')
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_ijepa_features.py'),
        '--device', str(device), '--model', 'vit_huge', '--patch-size', '14',
        '--image-size', '224', '--checkpoint', str(ijepa_checkpoint),
        '--output', str(ROOT / 'runs/ijepa_views.npz'),
    ], cwd=ROOT, check=True)
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_dino_features.py'),
        '--device', str(device), '--image-size', '224',
        '--output', str(ROOT / 'runs/dino_views.npz'),
    ], cwd=ROOT, check=True)
    from jepa_lab.evaluation import temporal_perturbation_summary
    pprint({
        'I-JEPA': temporal_perturbation_summary(ROOT / 'runs/ijepa_views.npz').as_dict(),
        'DINO': temporal_perturbation_summary(ROOT / 'runs/dino_views.npz').as_dict(),
    })
else:
    print('Atlandı. Checkpoint ve ağ erişimi hazır olduğunda RUN_DINO_IJEPA_VIEWS=True yapın.')

## M2 geçiş kontrolü

- DINO: iki view arasındaki temsil eşleşmesi; I-JEPA: context'ten konuma koşullu target latent prediction.
- Düşük loss tek başına collapse olmadığını kanıtlamaz.
- Ablation'lar aynı veri/seed/step bütçesiyle kıyaslanır.
- Küçük koşunun paper sıralamasını üretmemesi veri, model ve bütçe farkıyla açıklanır.
- Sonraki aşamaya geçmek için k-NN veya frozen linear probe'da random encoder'a göre ≥5 puan farkı gerçek Imagenette koşusunda göstermelisiniz.